# Neuroticism Data Preparation

This notebook contains the code used to clean and prepare neuroticism data for the practical project **"A Neural Signature of Neuroticism? Exploring the Functional Connectome to Predict Personality"** <br>
This pipeline handles the initial data processing required for measurement model testing and the subsequent integration into the final Structural Equation Model (SEM).

The raw data analyzed in this project were sourced from the **Human Connectome Project Young Adult (HCP-YA) 2025 data release**. To comply with the [HCP-YA Restricted Data Use Agreement](https://www.humanconnectome.org/study/hcp-young-adult/data-use-terms), the raw participant data cannot be shared publicly. Consequently, the CSV file read in here (`hcp_dummy_data.csv`) contains **simulated noise** rather than real participant responses. While the values are synthetic, the column names are identical to the 2025 HCP release, ensuring the code is fully executable for demonstration. Researchers with authorized access to the HCP data can run this notebook by replacing the dummy file with the official dataset available at the [BALSA (Brain Analysis Library of Spatial Maps and Atlases)](https://balsa.wustl.edu/).

In [ ]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

# Load restricted HCP data
original_df = pd.read_csv("hcp_dummy_data.csv")  # REPLACE PATH TO OFFICIAL DATA HERE
print(original_df.shape)

(1206, 15)


In [16]:
# Select only the relevant cols
nums = [1, 6, 11, 16, 21, 26, 31, 36, 41, 46, 51, 56]
neoraw_cols = [f"NEORAW_{i:02d}" for i in nums]

neuroticism_df = original_df[['Subject', 'NEOFAC_N', 'NEO-FFI_Compl'] + neoraw_cols].copy()
neuroticism_df = neuroticism_df.dropna() # drop all rows with any NaNs
print(neuroticism_df.shape)
print(neuroticism_df.head())

(1206, 15)
   Subject  NEOFAC_N  NEO-FFI_Compl NEORAW_01 NEORAW_06 NEORAW_11 NEORAW_16  \
0   438716        23           True         A         D        SA         N   
1   717713        21           True        SA         D         D        SA   
2   973742        25           True        SA         D         A         N   
3   483168        23           True         N        SA         A        SA   
4   879328        21           True        SA         A         D         N   

  NEORAW_21 NEORAW_26 NEORAW_31 NEORAW_36 NEORAW_41 NEORAW_46 NEORAW_51  \
0         N        SD        SD         N         N        SA         A   
1         N         A        SD        SD         A        SA         A   
2         A         N         N        SD         A         A        SA   
3        SD        SD         D        SA         A        SA         D   
4         N        SA         D         A         D         N        SD   

  NEORAW_56  
0         N  
1        SA  
2        SA  
3      

In [17]:
# Numerical coding of responses
neuroticism_data_for_SEM = neuroticism_df.copy()

# Mapping according to the NEO-FFI Manual
mapping_normal = {"SD":0,"D":1,"N":2,"A":3,"SA":4}
mapping_reversed = {"SD":4,"D":3,"N":2,"A":1,"SA":0}

nums_normal = [6,11,21,26,36,41,51,56]
neoraw_cols_normal = [f"NEORAW_{i:02d}" for i in nums_normal]
nums_reversed = [1,16,31,46]
neoraw_cols_reversed = [f"NEORAW_{i:02d}" for i in nums_reversed]

# Replace within the copied dataframe
neuroticism_data_for_SEM[neoraw_cols_normal] = neuroticism_data_for_SEM[neoraw_cols_normal].replace(mapping_normal)
neuroticism_data_for_SEM[neoraw_cols_reversed] = neuroticism_data_for_SEM[neoraw_cols_reversed].replace(mapping_reversed)

print(neuroticism_data_for_SEM.head())


   Subject  NEOFAC_N  NEO-FFI_Compl NEORAW_01 NEORAW_06 NEORAW_11 NEORAW_16  \
0   438716        23           True         1         1         4         2   
1   717713        21           True         0         1         1         0   
2   973742        25           True         0         1         3         2   
3   483168        23           True         2         4         3         0   
4   879328        21           True         0         3         1         2   

  NEORAW_21 NEORAW_26 NEORAW_31 NEORAW_36 NEORAW_41 NEORAW_46 NEORAW_51  \
0         2         0         4         2         2         0         3   
1         2         3         4         0         3         0         3   
2         3         2         2         0         3         1         4   
3         0         0         3         4         3         0         1   
4         2         4         3         3         1         2         0   

  NEORAW_56  
0         2  
1         4  
2         4  
3         3  
4   

In [ ]:
# Double-check numerical coding
all_neuroticism_cols = neoraw_cols_normal + neoraw_cols_reversed

# Calculate the sum of numerically coded neuroticism items
neuroticism_data_for_SEM['Calculated_N_Sum'] = neuroticism_data_for_SEM[all_neuroticism_cols].sum(axis=1, min_count=1)

# Check for mismatches between calculated sum and HCP provided factor score
mismatches = neuroticism_data_for_SEM[
    neuroticism_data_for_SEM['NEOFAC_N'] != neuroticism_data_for_SEM['Calculated_N_Sum']
]
print(len(mismatches))

0


In [19]:
# Drop columns that are unnecessary after numerical coding check
cols_to_drop = ["NEOFAC_N", "NEO-FFI_Compl", "Calculated_N_Sum"]
neuroticism_data_for_SEM = neuroticism_data_for_SEM.drop(columns=cols_to_drop)

print(neuroticism_data_for_SEM.columns)

Index(['Subject', 'NEORAW_01', 'NEORAW_06', 'NEORAW_11', 'NEORAW_16',
       'NEORAW_21', 'NEORAW_26', 'NEORAW_31', 'NEORAW_36', 'NEORAW_41',
       'NEORAW_46', 'NEORAW_51', 'NEORAW_56'],
      dtype='object')


In [20]:
# Save prepped df
output_file = "prepped_dummy_data_for_SEM.csv"
neuroticism_data_for_SEM.to_csv(output_file, index=False)
